# 00. Python Primer for Engineers
**Engineering Data Processing with Python** | one-day course | about 40 minutes

> **TRAINER / SOLUTION VERSION.** Every blank is filled in and the notebook has been run.


In [1]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## What this is for

A gentle start. If you have never written Python, that is fine: you will mostly run code and change small things.
If you have, go quickly and help your neighbour.

The pattern all day is **see one, do one**: an example cell shows how something works, then a **Your turn** cell
asks you to do the same thing with something slightly different.

## 1. Colab in two minutes

* Grey boxes are **code cells**. Click one and press **Shift + Enter** to run it.
* The output appears underneath. The number in `[ ]` shows the order cells ran in.
* Got in a muddle? **Runtime > Restart session**, then run from the top. You cannot break anything.

In [2]:
print("Hello from Colab")
2 + 2

Hello from Colab


4

## 2. Variables: names for values

Put units in the name (`current_a`, not `c`). Example: three-phase motor input power.

In [3]:
voltage_v = 400
current_a = 32
power_factor = 0.85

power_kw = 1.732 * voltage_v * current_a * power_factor / 1000
print(f"Motor input power: {power_kw:.1f} kW")

Motor input power: 18.8 kW


`f"...{power_kw:.1f}..."` puts the value into the text, rounded to 1 decimal place.

### Your turn 0.1
Pump hydraulic power = density x g x flow x head. The formula is written. Fill in the flow (**0.05** m³/s) and
the head (**30** m).

In [4]:
density_kg_m3 = 1000
g = 9.81
flow_m3_s = 0.05
head_m = 30

hydraulic_power_kw = density_kg_m3 * g * flow_m3_s * head_m / 1000
print(f"Hydraulic power: {hydraulic_power_kw:.1f} kW")

Hydraulic power: 14.7 kW


In [5]:
check(abs(hydraulic_power_kw - 14.715) < 0.01, "About 14.7 kW. Sounds like a small process pump.", "Expected about 14.7 kW.")

PASS: About 14.7 kW. Sounds like a small process pump.


## 3. Text, and the "15 kW" problem

Numbers that arrive as **text** cause most messy-data pain. Python cannot add 5 to `"15 kW"`:

In [6]:
rated_power = "15 kW"
try:
    rated_power + 5
except TypeError as err:
    print("Error:", err)

# Remove the unit, then turn the text into a number
print(float(rated_power.replace("kW", "")) + 5)

Error: can only concatenate str (not "int") to str
20.0


Text has built-in tools. `.strip()` removes spaces at the ends, `.upper()` makes capitals, `.lower()` makes lower case.

In [7]:
print(repr("  Cork  ".strip()))
print("cork".upper())

'Cork'
CORK


### Your turn 0.2
Turn `"  cnc-009 "` into `"CNC-009"`. Choose from: `strip`, `upper`, `lower`.

In [8]:
clean_id = "  cnc-009 ".strip().upper()
print(repr(clean_id))

'CNC-009'


In [9]:
check(clean_id == "CNC-009", "Clean ID.", "Expected 'CNC-009' with no spaces.")

PASS: Clean ID.


## 4. Lists: several values in order

In [10]:
bearing_temp_c = [61.2, 63.5, 62.8, 70.4, 66.1, 64.0]

print("How many:", len(bearing_temp_c))
print("Highest:", max(bearing_temp_c))
print("Lowest:", min(bearing_temp_c))
print("Average:", sum(bearing_temp_c) / len(bearing_temp_c))
print("First one:", bearing_temp_c[0])     # counting starts at 0

How many: 6
Highest: 70.4
Lowest: 61.2
Average: 64.66666666666667
First one: 61.2


### Your turn 0.3
Same again for vibration. Find the **highest** reading. Choose from: `len`, `max`, `min`, `sum`.

In [11]:
vibration_mm_s = [2.1, 2.4, 2.2, 7.8, 2.3, 9.4, 2.5, 8.1, 2.2]
highest = max(vibration_mm_s)
print("Highest vibration:", highest, "mm/s")

Highest vibration: 9.4 mm/s


In [12]:
check(highest == 9.4, "9.4 mm/s.", "Expected 9.4.")

PASS: 9.4 mm/s.


## 5. Loops and decisions

`for` repeats for each item. `if` makes a decision. The **indent** (4 spaces) shows which lines belong inside.

In [13]:
LIMIT_C = 65
for t in bearing_temp_c:
    if t > LIMIT_C:
        print(t, "ABOVE LIMIT")
    else:
        print(t, "ok")

61.2 ok
63.5 ok
62.8 ok
70.4 ABOVE LIMIT
66.1 ABOVE LIMIT
64.0 ok


### Your turn 0.4
Do the same for vibration with a limit of **7.1** mm/s. Fill in the list to loop over and the limit.

In [14]:
alarms = []
for v in vibration_mm_s:
    if v > 7.1:
        alarms.append(v)          # add this reading to the alarms list
print("Alarms:", alarms)

Alarms: [7.8, 9.4, 8.1]


In [15]:
check(alarms == [7.8, 9.4, 8.1], "Three alarm readings.", "Expected [7.8, 9.4, 8.1].")

PASS: Three alarm readings.


## 6. Dictionaries: values with labels

Think of one row in an asset register.

In [16]:
asset = {"asset_id": "CMP-001", "asset_type": "Air Compressor", "site": "Cork", "rated_power_kw": 55.0}
print(asset["site"])
asset["criticality"] = "A"      # add a new label and value
print(asset)

Cork
{'asset_id': 'CMP-001', 'asset_type': 'Air Compressor', 'site': 'Cork', 'rated_power_kw': 55.0, 'criticality': 'A'}


### Your turn 0.5
Print the asset's rated power. Copy the label exactly from the dictionary above.

In [17]:
print(asset["rated_power_kw"])

55.0


## 7. Functions: a reusable calculation

`def` creates a function, `return` gives back the answer.

In [18]:
def celsius_to_fahrenheit(temp_c):
    return temp_c * 9 / 5 + 32

print(celsius_to_fahrenheit(100))

212.0


### Your turn 0.6
Kelvin to Celsius: subtract **273.15**.

In [19]:
def kelvin_to_celsius(temp_k):
    return temp_k - 273.15

print(kelvin_to_celsius(353.15))

80.0


In [20]:
check(abs(kelvin_to_celsius(353.15) - 80) < 1e-9, "353.15 K is 80 °C.", "Subtract 273.15.")

PASS: 353.15 K is 80 °C.


## 8. A real file: a boiler data dictionary

`boiler.json` lists the variables in a boiler simulation export and their units. JSON is just dictionaries and lists
saved as text. Run and read.

In [21]:
import json
with open("data/boiler.json") as f:
    boiler = json.load(f)

for var in boiler["variables"]:
    print(var["Name"], "| export unit:", var["Unit"], "| model unit:", var["Modelica.Unit"])

Time_s | export unit: s | model unit: s
P_fuel_kW | export unit: kW | model unit: W
P_thermal_kW | export unit: kW | model unit: W
T_out_degC | export unit: degC | model unit: K
G_losses | export unit: - | model unit: -
eta | export unit: - | model unit: -


## 9. Working with Gemini in Colab

(Full guidance is in the exercise version.) Exercise 0.7: the bug is a misspelt name, `bearing_temps` instead of
`bearing_temp_c`, giving `NameError`.

In [22]:
average = sum(bearing_temp_c) / len(bearing_temp_c)
print(f"Average bearing temperature: {average:.1f} °C")

Average bearing temperature: 64.7 °C


## 10. First look at pandas

`pandas` is the Python tool for tables. A table is called a **DataFrame**. Run and read.

In [23]:
printer = pd.read_csv("data/3dprinter.csv")
printer.head()

,layer_height,wall_thickness,infill_density,infill_pattern,nozzle_temperature,bed_temperature,print_speed,material,fan_speed,roughness,tension_strenght,elongation
0,0.02,8,90,grid,220,60,40,abs,0,25,18.0,1.2
1,0.02,7,90,honeycomb,225,65,40,abs,25,32,16.0,1.4
2,0.02,1,80,grid,230,70,40,abs,50,40,8.0,0.8
3,0.02,4,70,honeycomb,240,75,40,abs,75,68,10.0,0.5
4,0.02,6,90,grid,250,80,40,abs,100,92,5.0,0.7


In [24]:
print(printer.shape)                           # (rows, columns)
printer["material"].value_counts()             # how many of each

(51, 12)


material
pla    26
abs    25
Name: count, dtype: int64

### Your turn 0.8
Count how many tests used each **infill_pattern** (a column name in the table above).

In [25]:
printer["infill_pattern"].value_counts()

infill_pattern
grid         26
honeycomb    25
Name: count, dtype: int64

## Recap

| You want to | Python |
|---|---|
| Store a value | `power_kw = 15.0` |
| Tidy text | `"  cnc-009 ".strip().upper()` |
| Highest / count | `max(values)`, `len(values)` |
| Repeat with a decision | `for v in values:` then `if v > 7.1:` |
| Read a CSV | `pd.read_csv("data/file.csv")` |
| Count values in a column | `df["column"].value_counts()` |

Next: **01 Engineering Data Reality Check**.